# Lab 11 — Image Processing as Linear Algebra
**Linear Algebra · UATX**  
*Weeks 10–11 · Prerequisites: §4.3, Lab 10 (SVD)*

An image is a matrix $I \in \mathbb{R}^{m \times n}$. Every classical image filter (blur, sharpen, edge-detect) is a **linear map**. In 1D, convolution with a kernel $h$ is multiplication by a **Toeplitz matrix** $T_h$. The SVD of $T_h$ reveals the filter's geometry: how it scales different "frequency" directions.

**Tasks**
1. Load a grayscale image; treat it as a matrix and a vector in $\mathbb{R}^{mn}$.
2. Build the Toeplitz matrix for 1D convolution; verify it matches `scipy.ndimage.convolve`.
3. Apply classic 2D kernels (blur, sharpen, Sobel, Laplacian) as matrix operations; verify linearity.
4. Compute the SVD of a blur Toeplitz matrix; interpret singular-value decay.
5. *(Extension)* Deconvolution: invert a blur filter via pseudoinverse; observe noise amplification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import datasets
from scipy.ndimage import convolve
from scipy.linalg import toeplitz

np.random.seed(42)

## §1  Images as Matrices

A grayscale image of height $m$ and width $n$ is simply a matrix $I \in \mathbb{R}^{m \times n}$, with each entry in $[0,1]$ representing pixel intensity. Equivalently, $\mathrm{vec}(I) \in \mathbb{R}^{mn}$ is a vector; the space of all such images is a vector space (addition and scalar multiplication work pixel-wise).

In [ ]:
# Load a standard grayscale test image
img = datasets.ascent().astype(float) / 255.0   # 512x512, values in [0,1]
m, n = img.shape
print(f'Image shape: {img.shape}  ({m*n} pixels)')
print(f'min={img.min():.3f}, max={img.max():.3f}, mean={img.mean():.3f}')
print(f'Type: {type(img)}, dtype: {img.dtype}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Image as matrix $I \\in \\mathbb{{R}}^{{{m} \\times {n}}}$')
axes[0].axis('off')

# Show a 10x10 patch as numbers
patch = img[250:260, 250:260]
im2 = axes[1].imshow(patch, cmap='gray', vmin=0, vmax=1)
for i in range(10):
    for j in range(10):
        axes[1].text(j, i, f'{patch[i,j]:.2f}', ha='center', va='center',
                     fontsize=6, color='red' if patch[i,j] < 0.5 else 'blue')
axes[1].set_title('10×10 patch: pixel values')
plt.tight_layout(); plt.show()

# Vector space operations on images
img2 = datasets.face(gray=True).astype(float) / 255.0
img2 = img2[:m, :n]   # crop to same size
blend = 0.5 * img + 0.5 * img2
print(f'\n0.5*img + 0.5*img2 still in [0,1]: {blend.min():.3f} to {blend.max():.3f}')

## §2  Convolution as Matrix Multiplication

In 1D, convolving a signal $x \in \mathbb{R}^n$ with a kernel $h \in \mathbb{R}^k$ gives
$(h * x)_i = \sum_j h_j\, x_{i-j}$.
This is **exactly** multiplication by a Toeplitz matrix $T_h \in \mathbb{R}^{n \times n}$ (with wrap-around / zero-padding).

The same structure extends to 2D via **block-Toeplitz** matrices, but we illustrate in 1D where we can display $T_h$ explicitly.

In [ ]:
def toeplitz_conv(h, n, mode='same'):
    """
    Build the n x n Toeplitz convolution matrix for kernel h.
    Uses zero-padding (mode='same'): output has the same length as input.
    """
    k = len(h)
    pad = k // 2
    # First column and first row of the Toeplitz matrix
    col = np.zeros(n)
    row = np.zeros(n)
    for i, hi in enumerate(h):
        if i - pad >= 0:
            row[i - pad] = hi
        elif pad - i < n:
            col[pad - i] = hi
    col[0] = h[pad]
    return toeplitz(col, row)


# --- 1D example ---
n1d = 8
x = np.array([0., 0., 0., 1., 0., 0., 0., 0.])   # impulse signal

# Box (averaging) kernel of width 3
h_box = np.array([1/3, 1/3, 1/3])
T_box = toeplitz_conv(h_box, n1d)

print('Toeplitz matrix for box-3 kernel (8x8):')
print(np.round(T_box, 3))

# Method 1: matrix multiplication
y_matrix = T_box @ x

# Method 2: scipy convolve (1D, mode='same')
y_scipy = convolve(x.reshape(1, -1), h_box.reshape(1, -1), mode='constant')[0]

print('\nMatrix multiplication result:', np.round(y_matrix, 4))
print('scipy convolve result:       ', np.round(y_scipy, 4))
print('Match:', np.allclose(y_matrix, y_scipy))

In [ ]:
# Visualise Toeplitz structure for a Gaussian kernel
def gaussian_kernel_1d(sigma, width):
    x = np.arange(-(width//2), width//2 + 1)
    g = np.exp(-x**2 / (2*sigma**2))
    return g / g.sum()

h_gauss = gaussian_kernel_1d(sigma=2, width=9)
T_gauss = toeplitz_conv(h_gauss, 30)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(np.arange(len(h_gauss)) - len(h_gauss)//2, h_gauss)
axes[0].set_title('Gaussian kernel $h$'); axes[0].set_xlabel('offset'); axes[0].grid(True)

axes[1].imshow(T_gauss, cmap='Blues', aspect='auto')
axes[1].set_title('Toeplitz matrix $T_h$ (30×30)'); axes[1].set_xlabel('column'); axes[1].set_ylabel('row')

# Impulse response = first nonzero column (should recover h)
imp = np.zeros(30); imp[15] = 1.0
axes[2].plot(T_gauss @ imp, 'o-', ms=4)
axes[2].set_title('Impulse response $T_h\\,\\delta = h$')
axes[2].grid(True); axes[2].set_xlabel('index')
plt.tight_layout(); plt.show()

## §3  Classic 2D Kernels as Linear Maps

Each image filter is a linear map $T: \mathbb{R}^{m\times n} \to \mathbb{R}^{m\times n}$, i.e., $T(\alpha I + \beta J) = \alpha T(I) + \beta T(J)$.
We implement a wrapper `apply_kernel` using `scipy.ndimage.convolve` and verify linearity numerically.

In [ ]:
def gaussian_kernel_2d(sigma=1.0, width=5):
    ax = np.arange(-(width//2), width//2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    g = np.exp(-(xx**2 + yy**2) / (2*sigma**2))
    return g / g.sum()

def apply_kernel(image, kernel):
    """Apply a 2D convolution kernel to a grayscale image."""
    return convolve(image, kernel, mode='reflect')


# Define standard kernels
kernels = {
    'Gaussian blur (σ=2)': gaussian_kernel_2d(sigma=2, width=9),
    'Sharpen'            : np.array([[ 0, -1,  0],
                                     [-1,  5, -1],
                                     [ 0, -1,  0]], float),
    'Sobel X'            : np.array([[-1,  0,  1],
                                     [-2,  0,  2],
                                     [-1,  0,  1]], float),
    'Sobel Y'            : np.array([[-1, -2, -1],
                                     [ 0,  0,  0],
                                     [ 1,  2,  1]], float),
    'Laplacian'          : np.array([[ 0,  1,  0],
                                     [ 1, -4,  1],
                                     [ 0,  1,  0]], float),
}

# Use a small crop for display speed
crop = img[128:384, 128:384]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(crop, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original'); axes[0, 0].axis('off')

for ax, (name, k) in zip(axes.flat[1:], kernels.items()):
    filtered = apply_kernel(crop, k)
    # For edge kernels, centre around 0.5 for display
    if 'Sobel' in name or 'Laplacian' in name:
        display = 0.5 + filtered / (2 * np.abs(filtered).max() + 1e-9)
    else:
        display = np.clip(filtered, 0, 1)
    ax.imshow(display, cmap='gray', vmin=0, vmax=1)
    ax.set_title(name); ax.axis('off')

plt.suptitle('Image filters as linear maps', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Verify linearity: T(alpha*I + beta*J) = alpha*T(I) + beta*T(J)
J = img2[:m, :n]
alpha, beta = 0.7, -0.3
kernel_test = kernels['Sharpen']

lhs = apply_kernel(alpha * img + beta * J, kernel_test)
rhs = alpha * apply_kernel(img, kernel_test) + beta * apply_kernel(J, kernel_test)

print('Linearity check for Sharpen kernel:')
print(f'  ||T(αI + βJ) - αT(I) - βT(J)||_∞ = {np.abs(lhs - rhs).max():.2e}')
print(f'  Linear: {np.allclose(lhs, rhs)}')

# Verify for all kernels
for name, k in kernels.items():
    lhs = apply_kernel(alpha*img + beta*J, k)
    rhs = alpha*apply_kernel(img, k) + beta*apply_kernel(J, k)
    print(f'  {name:<25} linear: {np.allclose(lhs, rhs)}')

## §4  SVD of a Convolution Matrix

The Toeplitz matrix $T_h$ has a well-defined SVD. For a blur kernel, the singular values decay rapidly: blurring maps most of the "energy" into a low-dimensional subspace, which is precisely why blurred images are compressible (Lab 10).

We also compute the **Sobel** Toeplitz matrix and see that its singular-value profile is very different — edge detectors amplify high-frequency content.

In [ ]:
N = 64   # work with 64-point signals for speed

# Build Toeplitz matrices for three kernels
h_blur   = gaussian_kernel_1d(sigma=3, width=13)
h_sharp  = np.array([-1/3, 1+2/3, -1/3])   # 1D sharpening
h_diff   = np.array([-0.5, 0., 0.5])        # central difference (derivative)

T_blur  = toeplitz_conv(h_blur,  N)
T_sharp = toeplitz_conv(h_sharp, N)
T_diff  = toeplitz_conv(h_diff,  N)

# SVD of each
svd_results = {}
for name, T in [('Gaussian blur', T_blur), ('Sharpen', T_sharp), ('Derivative', T_diff)]:
    _, s, _ = np.linalg.svd(T)
    svd_results[name] = s
    print(f'{name:<20} top-5 sv: {np.round(s[:5], 4)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, s in svd_results.items():
    axes[0].plot(s, label=name)
    axes[1].semilogy(s + 1e-15, label=name)

for ax in axes:
    ax.set_xlabel('Index $i$'); ax.legend(); ax.grid(True)
axes[0].set_ylabel('$\\sigma_i$'); axes[0].set_title('Singular values of Toeplitz matrices')
axes[1].set_ylabel('$\\sigma_i$ (log scale)'); axes[1].set_title('Log scale')
plt.tight_layout(); plt.show()

# Blur has fast decay -> low effective rank
tol = 1e-3
for name, s in svd_results.items():
    eff_rank = np.sum(s / s[0] > tol)
    print(f'{name:<20}  effective rank (σ_i/σ_1 > {tol}): {eff_rank}/{N}')

In [ ]:
# Singular vectors of T_blur: they look like sinusoids (Fourier modes)
_, s_b, Vt_b = np.linalg.svd(T_blur)

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    ax.plot(Vt_b[i], lw=1.5)
    ax.set_title(f'$v_{{{i+1}}}$, $\\sigma={s_b[i]:.3f}$')
    ax.set_ylim(-0.25, 0.25); ax.grid(True)
plt.suptitle('Right singular vectors of Gaussian blur Toeplitz matrix')
plt.tight_layout(); plt.show()

print('Observation: singular vectors of a blur matrix are (approximately) Fourier modes.')
print('Blur strongly attenuates high-frequency modes (small σ_i for large i).')

## §5  Gradient Magnitude: Combining Linear Maps

The **Sobel edge detector** computes the gradient magnitude $\|\nabla I\|_2 = \sqrt{(T_{Sx}\, \mathrm{vec}(I))^2 + (T_{Sy}\,\mathrm{vec}(I))^2}$ pixel-wise. The gradient magnitude map is **not** itself a linear map (because of the $\sqrt{\cdot}$), but each component $T_{Sx}$ and $T_{Sy}$ is.

We also illustrate that composing linear filters = composing linear maps: $T_{S_x} \circ T_{\text{blur}}$ first blurs, then differentiates.

In [ ]:
# Apply to the full image
blur_img  = apply_kernel(img, kernels['Gaussian blur (σ=2)'])
sobel_x   = apply_kernel(blur_img, kernels['Sobel X'])
sobel_y   = apply_kernel(blur_img, kernels['Sobel Y'])
grad_mag  = np.sqrt(sobel_x**2 + sobel_y**2)
grad_dir  = np.arctan2(sobel_y, sobel_x)   # angle in [-pi, pi]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(blur_img, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Blur ($T_{\\mathrm{blur}}$)'); axes[1].axis('off')
axes[2].imshow(grad_mag, cmap='hot'); axes[2].set_title('$\\|\\nabla I\\|$ (edge strength)'); axes[2].axis('off')
axes[3].imshow(grad_dir, cmap='hsv'); axes[3].set_title('$\\angle \\nabla I$ (edge direction)'); axes[3].axis('off')
plt.suptitle('Blur → Sobel pipeline: $T_{S_x} \\circ T_{\\mathrm{blur}}$', fontsize=12)
plt.tight_layout(); plt.show()

# Show that blur then Sobel ≠ Sobel then blur (convolution is commutative; order here matters only in practice)
sobel_first = apply_kernel(apply_kernel(img, kernels['Sobel X']), kernels['Gaussian blur (σ=2)'])
blur_first  = apply_kernel(apply_kernel(img, kernels['Gaussian blur (σ=2)']), kernels['Sobel X'])
print('For convolution kernels, order commutes in theory.')
print(f'||T_blur T_Sx - T_Sx T_blur||_F = {np.linalg.norm(sobel_first - blur_first):.2e}')

## Extension — Deconvolution via Pseudoinverse

If $y = T_h\, x$ (blur applied to signal), can we recover $x$?  
The pseudoinverse $T_h^+$ gives the least-norm solution, but when singular values of $T_h$ are tiny, the pseudoinverse blows up — **noise is amplified**. This is an ill-posed inverse problem.

We demonstrate this in 1D with a clean and a noisy blurred signal.

In [ ]:
N_dec = 64
# Ground-truth signal: a piecewise-constant function
x_true = np.zeros(N_dec)
x_true[10:20] = 1.0
x_true[35:50] = 0.6
x_true[55:60] = 0.8

# Build blur Toeplitz
h_b = gaussian_kernel_1d(sigma=2, width=9)
T_b = toeplitz_conv(h_b, N_dec)

# Blurred signal, clean and noisy
y_clean = T_b @ x_true
y_noisy = y_clean + 0.005 * np.random.randn(N_dec)

# Pseudoinverse via SVD (with different truncation thresholds)
U_b, s_b2, Vt_b2 = np.linalg.svd(T_b)

def pinv_truncated(U, s, Vt, rcond):
    s_inv = np.where(s > rcond * s.max(), 1.0/s, 0.0)
    return (Vt.T * s_inv) @ U.T

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Deconvolution: pseudoinverse of blur Toeplitz matrix', fontsize=12)

for row, (y, label) in enumerate([(y_clean, 'Clean'), (y_noisy, 'Noisy (σ=0.005)')]):
    for col, rcond in enumerate([1e-1, 1e-3, 1e-10]):
        T_pinv = pinv_truncated(U_b, s_b2, Vt_b2, rcond)
        x_rec = T_pinv @ y
        err = np.linalg.norm(x_rec - x_true)
        ax = axes[row, col]
        ax.plot(x_true, 'k-', lw=2, label='true')
        ax.plot(x_rec, 'r-', lw=1.5, alpha=0.8, label=f'rcond={rcond}')
        ax.set_title(f'{label}, rcond={rcond}\nerr={err:.3f}')
        ax.legend(fontsize=7); ax.set_ylim(-2, 3); ax.grid(True)

plt.tight_layout(); plt.show()

print('Observation: small rcond keeps more singular values → better fit on clean signal,')
print('but dramatically amplifies noise. This is the core difficulty of deconvolution.')